## 1. Importación de librerías

In [82]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from mlflow.models.signature import infer_signature

import mlflow
import mlflow.sklearn


## 2. Configuración de MLflow



In [83]:
mlflow.set_tracking_uri("http://127.0.0.1:9090")
mlflow.set_experiment("arbolestudiantes")


2026/05/20 12:49:21 INFO mlflow.tracking.fluent: Experiment with name 'arbolestudiantes' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1779299361651, experiment_id='4', last_update_time=1779299361651, lifecycle_stage='active', name='arbolestudiantes', tags={}, trace_location=None, workspace='default'>

## 3. Carga del dataset

In [84]:
df = pd.read_csv("data/estudiantes.csv")
df.head()


,carrera,modalidad,beca,edad,promedio,asistencias,aprobado
0,Industrial,Presencial,Si,29,5.8,64,Si
1,Industrial,Hibrida,Si,27,6.6,51,No
2,Arquitectura,Presencial,Si,29,8.2,84,Si
3,Economia,Presencial,Si,29,6.6,67,No
4,Economia,Presencial,Si,24,5.1,72,No


In [85]:
print("Filas y columnas:", df.shape)
df.info()


Filas y columnas: (5000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   carrera      5000 non-null   object 
 1   modalidad    5000 non-null   object 
 2   beca         5000 non-null   object 
 3   edad         5000 non-null   int64  
 4   promedio     5000 non-null   float64
 5   asistencias  5000 non-null   int64  
 6   aprobado     5000 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 273.6+ KB


## 4. Preparación de datos

La variable objetivo `y` se transforma:

- `yes` → `1`
- `no` → `0`

Además, se elimina `duration` de las variables predictoras porque normalmente se conoce después de realizada la llamada. Por tanto, puede generar fuga de información si se usa para predecir antes de contactar al cliente.


In [86]:
df["aprobado"] = df["aprobado"].map({"Si": 1, "No": 0})
#print(df["aprobado"].unique())

X = df.drop(columns=["aprobado"])
y = df["aprobado"]

#print("Columnas originales usadas para entrenamiento:")
#print(df.columns.tolist())

#print("\nDistribución de la variable objetivo:")
#print(y.value_counts())


## 5. Identificación de columnas categóricas y numéricas

El pipeline hará automáticamente el `OneHotEncoder` para las columnas categóricas y dejará pasar las columnas numéricas.


In [87]:
columnas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()
columnas_numericas = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Columnas categóricas:")
print(columnas_categoricas)

print("\nColumnas numéricas:")
print(columnas_numericas)


Columnas categóricas:
['carrera', 'modalidad', 'beca']

Columnas numéricas:
['edad', 'promedio', 'asistencias']


In [88]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [89]:
#Preposesamiento
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas)
    ]
)

## 7. Creación del Pipeline

In [90]:
# Parámetros del árbol
# =====================================================
# 10 ITERACIONES CON DIFERENTE random_state
# =====================================================

iteraciones = [

    {
        "criterion": "gini",
        "max_depth": 3,
        "min_samples_split": 40,
        "min_samples_leaf": 10
    },

    {
        "criterion": "gini",
        "max_depth": 4,
        "min_samples_split": 60,
        "min_samples_leaf": 20
    },

    {
        "criterion": "gini",
        "max_depth": 5,
        "min_samples_split": 80,
        "min_samples_leaf": 30
    },

    {
        "criterion": "gini",
        "max_depth": 5,
        "min_samples_split": 100,
        "min_samples_leaf": 40
    },

    {
        "criterion": "gini",
        "max_depth": 6,
        "min_samples_split": 120,
        "min_samples_leaf": 50
    },

    {
        "criterion": "gini",
        "max_depth": 7,
        "min_samples_split": 140,
        "min_samples_leaf": 60
    },

    {
        "criterion": "gini",
        "max_depth": 8,
        "min_samples_split": 160,
        "min_samples_leaf": 70
    },

    {
        "criterion": "gini",
        "max_depth": 9,
        "min_samples_split": 180,
        "min_samples_leaf": 80
    },

    {
        "criterion": "gini",
        "max_depth": 10,
        "min_samples_split": 200,
        "min_samples_leaf": 90
    },

    {
        "criterion": "gini",
        "max_depth": 12,
        "min_samples_split": 220,
        "min_samples_leaf": 100
    }

]

In [91]:
# =====================================================
# CERRAR RUNS ACTIVOS
# =====================================================

while mlflow.active_run() is not None:
    mlflow.end_run()

# =====================================================
# ITERACIONES
# =====================================================
for i, params in enumerate(iteraciones, start=1):

    with mlflow.start_run(run_name=f"iteracion_{i}") as run:

        print(f"Ejecutando iteración {i}")

        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("modelo", DecisionTreeClassifier(
                    criterion=params["criterion"],
                    max_depth=params["max_depth"],
                    min_samples_split=params["min_samples_split"],
                    min_samples_leaf=params["min_samples_leaf"],
                    random_state=42
                ))
            ]
        )

        pipeline

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        # Parámetros del algoritmo
        mlflow.log_param("iteracion", i)
        mlflow.log_param("criterion", params["criterion"])
        mlflow.log_param("max_depth", params["max_depth"])
        mlflow.log_param("min_samples_split", params["min_samples_split"])
        mlflow.log_param("min_samples_leaf", params["min_samples_leaf"])
       # mlflow.log_param("random_state", params["random_state"])

        # Métricas
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        # Firma del modelo
        input_example = X_train.head(5)

        signature = infer_signature(
            input_example,
            pipeline.predict(input_example)
        )

        # Guardar modelo
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path=f"modelo_iteracion_{i}",
            registered_model_name="Modelo_estudiantes",
            input_example=input_example,
            signature=signature
        )

        print(f"Iteración {i}")
        print("Accuracy:", accuracy)
        print("F1:", f1)


Ejecutando iteración 1


C:\Users\PC-PR\Documents\herramientastia\entornoia3\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/20 12:49:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/20 12:49:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requ

Registered model 'Modelo_estudiantes' already exists. Creating a new version of this model...
2026/05/20 12:50:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Modelo_estudiantes, version 1
Created version '1' of model 'Modelo_estudiantes'.


Iteración 1
Accuracy: 0.858
F1: 0.8233830845771144
🏃 View run iteracion_1 at: http://127.0.0.1:9090/#/experiments/4/runs/8e343eed748e4beea21189eef581deff
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/4
Ejecutando iteración 2


C:\Users\PC-PR\Documents\herramientastia\entornoia3\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/20 12:50:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/20 12:50:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requ

...

Registered model 'Modelo_estudiantes' already exists. Creating a new version of this model...
2026/05/20 12:51:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Modelo_estudiantes, version 9
Created version '9' of model 'Modelo_estudiantes'.


Iteración 9
Accuracy: 0.851
F1: 0.807741935483871
🏃 View run iteracion_9 at: http://127.0.0.1:9090/#/experiments/4/runs/a04c969fe5e040c4b1edc2c0f4edbf11
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/4
Ejecutando iteración 10


C:\Users\PC-PR\Documents\herramientastia\entornoia3\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/20 12:51:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/20 12:51:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requ

Registered model 'Modelo_estudiantes' already exists. Creating a new version of this model...
2026/05/20 12:51:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Modelo_estudiantes, version 10
Created version '10' of model 'Modelo_estudiantes'.


Iteración 10
Accuracy: 0.851
F1: 0.807741935483871
🏃 View run iteracion_10 at: http://127.0.0.1:9090/#/experiments/4/runs/31d28968438f466ca88947ec8c67f881
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/4
